# 37 — Operating Regimes Improve Admissibility

**Engineering question**

Once a system can generate resources and allocate effort, where does the architecture actually work?

Notebook 31 established:

```text
physics → generated resources → admissibility → engineering allocation
```

Notebook 37 adds the next engineering layer:

```text
admitted resources → stable operating regime → usable computation
```

Engineering statement:

> Engineering seeks operating regimes where the greatest number of leading specifications remain simultaneously admitted.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/37_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def normalize(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Operating window

The goal is not to maximize squeezing in isolation or minimize loss in isolation.

The engineering goal is to maintain an operating window where the resource is simultaneously useful, stable, and measurable.

This figure uses an illustrative admissibility score:

```text
admissibility = squeezing benefit × low-loss benefit × stability envelope
```


In [ ]:
s = np.linspace(0, 1, 220)   # squeezing / interaction strength
l = np.linspace(0, 1, 220)   # optical loss
S, L = np.meshgrid(s, l)

# Conceptual regime score.
squeezing_benefit = 1 / (1 + np.exp(-14 * (S - 0.34)))
loss_penalty = np.exp(-3.2 * L)
overdrive_penalty = np.exp(-5.0 * np.maximum(S - 0.82, 0) ** 2)
thermal_penalty = np.exp(-4.0 * np.maximum(S + 0.8 * L - 1.12, 0) ** 2)

score = squeezing_benefit * loss_penalty * overdrive_penalty * thermal_penalty
score = score / score.max()

fig, ax = plt.subplots(figsize=(8.5, 5.4))
im = ax.imshow(score, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0, vmax=1)

contours = ax.contour(S, L, score, levels=[0.35, 0.55, 0.72], colors="black", linewidths=[1.2, 1.6, 2.2])
ax.clabel(contours, inline=True, fontsize=8)

ax.scatter([0.58], [0.23], s=120)
ax.text(0.58, 0.28, "largest admissible\noperating region", ha="center", fontsize=10)

ax.text(0.17, 0.82, "insufficient\nsqueezing", ha="center", fontsize=10)
ax.text(0.82, 0.82, "unstable /\noverdriven", ha="center", fontsize=10)
ax.text(0.82, 0.12, "useful but\nloss-limited", ha="center", fontsize=10)

ax.set_title("Operating Window: Maintain the Admitted Regime", fontsize=16)
ax.set_xlabel("Squeezing / nonlinear interaction strength")
ax.set_ylabel("Optical loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("admissibility score")

savefig(fig, "37_01_operating_window.png")
plt.show()


## 2. Constraint intersection

A usable operating regime is the common intersection of many constraints.

Each single constraint may look feasible by itself.

The architecture works only where all relevant constraints overlap.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_aspect("equal")
ax.axis("off")

constraint_names = [
    ("loss", (-0.18, 0.05), 0.48),
    ("graph", (0.18, 0.05), 0.48),
    ("timing", (0.00, 0.36), 0.48),
    ("detection", (-0.05, -0.28), 0.48),
    ("calibration", (0.30, -0.20), 0.48),
]

for name, center, radius in constraint_names:
    circ = Circle(center, radius, fill=False, linewidth=2.2, alpha=0.85)
    ax.add_patch(circ)
    ax.text(center[0], center[1] + radius + 0.055, name, ha="center", fontsize=11)

ax.add_patch(Circle((0.05, 0.02), 0.16, fill=True, alpha=0.2))
ax.text(0.05, 0.02, "admitted\noperating\nregion", ha="center", va="center", fontsize=10)

ax.set_xlim(-0.85, 0.9)
ax.set_ylim(-0.85, 0.95)
ax.set_title("Constraint Intersection: Operation Requires Overlap", fontsize=16)
ax.text(0.02, -0.78, "useful operation exists only inside the common intersection", ha="center", fontsize=10)

savefig(fig, "37_02_constraint_intersection.png")
plt.show()


## 3. Specification budget

A regime has a specification budget.

Some engineering work is required, some is optional, and some does not matter in a given regime.

This is not a dollar budget. It is a map of which engineering categories are active specifications.


In [ ]:
categories = ["physics", "engineering", "calibration", "feedback", "timing", "readout"]
required = np.array([0.55, 0.45, 0.35, 0.22, 0.25, 0.40])
optional = np.array([0.25, 0.35, 0.30, 0.34, 0.30, 0.28])
irrelevant = 1 - required - optional

x = np.arange(len(categories))
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.bar(x, required, label="required")
ax.bar(x, optional, bottom=required, label="optional")
ax.bar(x, irrelevant, bottom=required + optional, label="inactive in this regime")

ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("relative specification share")
ax.set_title("Specification Budget: Not Every Constraint Is Active", fontsize=16)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.25)

budget = pd.DataFrame({
    "category": categories,
    "required": required,
    "optional": optional,
    "inactive_in_this_regime": irrelevant,
})
budget.to_csv(CSV / "37_specification_budget.csv", index=False)
budget.to_json(JS / "37_specification_budget.json", orient="records", indent=2)

savefig(fig, "37_03_specification_budget.png")
plt.show()
budget


## 4. Regime transition

Admissibility and operating stability are related but different.

A resource can be generated and admitted in principle, while still being unusable because it falls outside the stable operating regime.


In [ ]:
labels = [
    "Generated\nresources",
    "Admitted\nresources",
    "Stable\noperating regime",
    "Fault-tolerant\nexecution",
    "Logical\ncomputation",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(11, 3.7))
ax.axis("off")

for i, label in enumerate(labels):
    rect = Rectangle((i - 0.46, -0.24), 0.92, 0.48, fill=False, linewidth=2.0)
    ax.add_patch(rect)
    ax.text(i, 0, label, ha="center", va="center", fontsize=9.5)
    if i < len(labels) - 1:
        ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.46, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

stage_notes = [
    "physics",
    "specification",
    "stability",
    "architecture",
]
for i, note in enumerate(stage_notes):
    ax.text(i + 0.5, 0.42, note, ha="center", fontsize=9)

ax.text((len(labels) - 1) / 2, -0.56, "admitted resources are not yet a stable operating regime", ha="center", fontsize=10.5)
ax.set_xlim(-0.72, len(labels) - 0.28)
ax.set_ylim(-0.75, 0.68)
ax.set_title("Regime Transition: Admitted Is Not Yet Stable", fontsize=16)

savefig(fig, "37_04_regime_transition.png")
plt.show()


## 5. Tradeoff surface

Single-variable optimization can be misleading.

The usable regime usually lies on a surface across loss, entanglement strength, and measurement quality.

The goal is to maintain the broadest stable region, not chase a single maximum.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

loss = np.linspace(0, 1, 50)
ent = np.linspace(0, 1, 50)
L, E = np.meshgrid(loss, ent)

# Measurement quality required rises with loss and very low entanglement.
M = np.clip(0.20 + 0.55 * L + 0.30 * (1 - E) + 0.10 * np.sin(2.5 * E), 0, 1)

# Regime score: high when entanglement is sufficient, loss is low, and measurement quality is adequate.
Z = (1 - L) * (0.25 + 0.75 * E) * (1 - 0.35 * np.abs(M - 0.55))
Z = Z / Z.max()

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(L, E, Z, linewidth=0, antialiased=True, alpha=0.92)

ax.set_title("Tradeoff Surface: Preserve the Operating Region", fontsize=15, pad=18)
ax.set_xlabel("loss")
ax.set_ylabel("entanglement strength")
ax.set_zlabel("admissibility")
ax.view_init(elev=28, azim=-55)

savefig(fig, "37_05_tradeoff_surface.png")
plt.show()


## 6. Operating-regime summary

The notebook's key distinction is:

```text
admitted resource ≠ stable operating regime
```

The operating regime is where the leading specifications remain satisfied together.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Stage": "Physics",
            "Question": "What can be generated?",
            "Maintained quantity": "Optical modes and nonlinear interaction",
            "Failure mode": "Insufficient resource generation",
        },
        {
            "Stage": "Admissibility",
            "Question": "What survives the constraints?",
            "Maintained quantity": "Admitted modes and graph edges",
            "Failure mode": "Resources fail the next specification",
        },
        {
            "Stage": "Engineering allocation",
            "Question": "Where should effort go?",
            "Maintained quantity": "Priority on bottleneck specifications",
            "Failure mode": "Effort spent on non-critical resources",
        },
        {
            "Stage": "Operating regime",
            "Question": "Where do specifications remain satisfied?",
            "Maintained quantity": "Stable intersection of constraints",
            "Failure mode": "Architecture leaves the admitted window",
        },
        {
            "Stage": "Computation",
            "Question": "Which logical operations are possible?",
            "Maintained quantity": "Fault-tolerant logical path",
            "Failure mode": "Logical depth or fidelity collapses",
        },
    ]
)

fig, ax = plt.subplots(figsize=(14, 4.2))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.02, 1.75)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Operating Regimes Preserve Admissibility", fontsize=16, pad=20)

summary.to_csv(CSV / "37_summary.csv", index=False)
summary.to_json(JS / "37_summary.json", orient="records", indent=2)

savefig(fig, "37_06_summary_table.png")
plt.show()
summary


## 7. Operating-regime map across architecture choices

Different architecture choices can satisfy different portions of the specification space.

A useful map shows not only whether a design works, but which regime it preserves.


In [ ]:
architectures = ["single resonator", "coupled resonators", "programmable mesh", "hybrid chip"]
specs = ["loss", "squeezing", "mode control", "detection", "feed-forward", "stability"]

values = np.array([
    [0.75, 0.55, 0.35, 0.55, 0.25, 0.70],
    [0.62, 0.70, 0.55, 0.50, 0.30, 0.58],
    [0.45, 0.65, 0.90, 0.62, 0.55, 0.45],
    [0.55, 0.70, 0.78, 0.75, 0.82, 0.60],
])

fig, ax = plt.subplots(figsize=(8.8, 4.8))
im = ax.imshow(values, vmin=0, vmax=1, aspect="auto")
ax.set_title("Architecture Regime Map", fontsize=16)
ax.set_xticks(np.arange(len(specs)))
ax.set_yticks(np.arange(len(architectures)))
ax.set_xticklabels(specs, rotation=25, ha="right")
ax.set_yticklabels(architectures)

for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        ax.text(j, i, f"{values[i,j]:.2f}", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("relative regime support")

arch_df = pd.DataFrame(values, index=architectures, columns=specs)
arch_df.to_csv(CSV / "37_architecture_regime_map.csv")
arch_df.to_json(JS / "37_architecture_regime_map.json", orient="index", indent=2)

savefig(fig, "37_07_architecture_regime_map.png")
plt.show()
arch_df


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available. For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "37_outputs.zip"

files_to_zip = [
    FIG / "37_01_operating_window.png",
    FIG / "37_02_constraint_intersection.png",
    FIG / "37_03_specification_budget.png",
    FIG / "37_04_regime_transition.png",
    FIG / "37_05_tradeoff_surface.png",
    FIG / "37_06_summary_table.png",
    FIG / "37_07_architecture_regime_map.png",
    CSV / "37_specification_budget.csv",
    CSV / "37_summary.csv",
    CSV / "37_architecture_regime_map.csv",
    JS / "37_specification_budget.json",
    JS / "37_summary.json",
    JS / "37_architecture_regime_map.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Generated resources are not enough.
- Admitted resources are still not enough.
- The architecture must preserve a stable operating regime.
- A stable operating regime is the intersection of loss, squeezing, measurement, timing, control, and calibration specifications.
- Engineering effort should maintain the region where those specifications remain jointly satisfied.

## Next notebook

**43 — Robust Operating Regions**

Notebook 43 can ask:

```text
How much can the system drift before the admitted operating regime fails?
```

That naturally extends Notebook 37 from operating-regime identification to robustness.
